# Data Exploration

First, the usual setup.

In [ ]:
from odyn import Database, Group

db = Database(r"tmp\test_server")

Now, we can check some of the data in the database. 

Each DB table has a corresponding `db.table_name` that returns a `pandas` `DataFrame` with the relevant data.

In [ ]:
db.experiments

In [ ]:
# You can also get particular columns like this
db.experiments.exp_name

In [ ]:
# Note that the "id" column of that table is a little different
#   (db.experiment.exp_id will not work!)
db.experiments.index

## Filtering Data

You can use _Data Wrangler_ or the `pandas` interface to filter tables.

In [ ]:
from datetime import datetime

# Next command == give me all experiments such that:
# - exp_start is at least 2025-10-10
# - mouse_id contains "237"
# You need parentheses for multiple conditions

db.experiments[
    (db.experiments.exp_start >= datetime(2025, 10, 10))
    & (db.experiments.mouse_id.str.contains("237"))
]

In [ ]:
# loc is to get a row by its index
db.experiments.loc[102]

## Pandas Example

Here are a simple example illustrating some `pandas` features.

You can check their documentation for a [quick reference](https://pandas.pydata.org/docs/getting_started/intro_tutorials/).

In [ ]:
# Select fine/coarse programs
program_names = ['fine 1', 'coarse 1', 'fine 2', 'coarse 2']
programs = db.programs[db.programs.program_type.isin(program_names)]

programs

In [ ]:
# Select non-passive trials with odors 17 and 18
odor_ids = [17, 18]
trials = db.trials[(db.trials.outcome != "na") & (db.trials.odor_id.isin(odor_ids))]

trials

In [ ]:
# Join a subset of columns from trials with one column from programs
result = trials[["program_id", "odor_id", "outcome"]].join(programs.program_type, on="program_id", how="inner")

result

In [ ]:
# Plot how many trials per outcome and odor
for odor_id in odor_ids:
    ax = result[result.odor_id == odor_id].pivot_table(
        index="program_type",
        columns="outcome",
        values="odor_id",
        aggfunc="count"
    ).plot(kind="barh")

    ax.set(title=f"Odor {odor_id}", ylabel="Program", xlabel="Trial Count")
